# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and analyze the [FAIR^2 Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and contains examined protein data from mass spectrometry experiments with annotated metadata. This notebook uses the schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset (metadata + records accessible via Croissant spec)
dataset = mlc.Dataset(croissant_url)

# Print high-level description
print(f"Title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Examine all available record sets, fields, and columns with their Croissant `@id` values.

In [ ]:
# List all record sets and associated fields with Croissant @id
record_sets = dataset.metadata.recordSets
print(f"Found {len(record_sets)} record sets.\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, dataType: {field.dataType}")
    print(f"  Columns (File mappings):")
    if rs.columns:
        for col in rs.columns:
            print(f"    - @id: {col.id}, name: {col.name}, source: {col.source}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we list all available record set `@id`s in this dataset and load each to pandas for exploration.

In [ ]:
# Choose record set ids dynamically
record_set_ids = [rs.id for rs in dataset.metadata.recordSets]
dataframes = {}

if len(record_set_ids) == 0:
    print("No record sets found in this dataset.")
else:
    for record_set_id in record_set_ids:
        print(f"Loading records for record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")
    # Use the first record set for further exploration
    main_record_set_id = record_set_ids[0]
    print(f"\nFirst few rows of first record set ({main_record_set_id}):")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter records, normalize numeric fields, and group data. All field references use Croissant field `@id` as shown in Data Overview.

In [ ]:
# For demonstration, extract a numeric and a categorical (group) field, by Croissant @id.
main_df = dataframes[main_record_set_id]

# Dynamically select a numeric field: find one with float or int in its dtype (fallback to known names if none)
numeric_field_id = None
for rs in dataset.metadata.recordSets:
    if rs.id == main_record_set_id:
        for field in rs.fields:
            if '\"float\"' in str(field.dataType).lower() or '\"number\"' in str(field.dataType).lower() or '\"integer\"' in str(field.dataType).lower():
                if field.id in main_df.columns:
                    numeric_field_id = field.id
                    break
if numeric_field_id is None:
    # Pick the first numeric-looking column
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break
if numeric_field_id is None:
    print("No numeric field detected for EDA. Aborting this section.")
else:
    print(f"Using numeric field: {numeric_field_id}")
    # Set threshold dynamically: 75th percentile
    threshold = main_df[numeric_field_id].quantile(0.75)
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())
    # Pick a group field (categorical): first non-numeric field
    group_field_id = None
    for col in main_df.columns:
        if col != numeric_field_id and not pd.api.types.is_numeric_dtype(main_df[col]):
            group_field_id = col
            break
    if group_field_id:
        print(f"Grouping by: {group_field_id}\n")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index(name=f"mean_{numeric_field_id}")
        display(grouped_df.head())
    else:
        print("No group (categorical) field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized version; if grouping was possible, show means per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    fig, ax = plt.subplots(1,2, figsize=(13,4))
    sns.histplot(main_df[numeric_field_id], kde=True, ax=ax[0])
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    if norm_col in filtered_df.columns:
        sns.histplot(filtered_df[norm_col], kde=True, ax=ax[1], color='orange')
        ax[1].set_title(f"Filtered & Normalized {numeric_field_id}")
    plt.tight_layout()
    plt.show()

    if 'grouped_df' in locals() and group_field_id is not None:
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=group_field_id, y=f"mean_{numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (filtered)")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to:

- Load, inspect, and analyze a Croissant Dataset using `mlcroissant`.
- Reference all data entities by their Croissant `@id`.
- Perform simple EDA: select numeric fields, filter, normalize, and group data.
- Visualize key fields.

For further in-depth biological analysis, refer to associated field definitions and protein annotations in the dataset schema and linked documentation.